# Transformer kernel — pass 2 follow-up (Tesla T4)

Short battery (~35 min) for what the first pass-2 run left open: the two gates
(their output wasn't captured last time), the `--dispatch` crash fix, the exact
`configs/best.json` submission configuration end to end, the `--fp16-max-elements`
cost on case 6, and CUDA graphs on the two untried mid shapes.
Already banked and NOT rerun here: the full default sweep, ablations, stress
batteries, and both case-14 runs (all committed under `results/`).


In [ ]:
!nvidia-smi --query-gpu=name,temperature.gpu,clocks.sm --format=csv
!cd /kaggle/working && rm -rf transformer-kernel && \
  git clone -q -b claude/integration-pass2 https://github.com/danielfodgaard/transformer-kernel.git
import torch
print(torch.__version__, torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))


## Gates (confirming on the record this time)


In [ ]:
!cd /kaggle/working/transformer-kernel && python src/test_kernels.py


In [ ]:
!cd /kaggle/working/transformer-kernel && python src/run_case.py --causal --batch-size 16 --d-model 128 --heads 4 \
  --seq-len 128 --layers 4 --ffn-dim 128 --padding-ratio 0.3 --warmup 0 --repeats 1 --benchmark-rounds 1


## The submission configuration, end to end

Exactly what `configs/best.json` encodes: CUDA graphs on cases 1–4 and 12, plain
defaults elsewhere. Expect ≈ the 7.11x best-config geomean.


In [ ]:
!cd /kaggle/working/transformer-kernel && python src/sweep.py --config configs/best.json --skip 14 --out results/best-verify.json


## --dispatch, post type-coercion fix (crashed last run)


In [ ]:
!cd /kaggle/working/transformer-kernel && python src/dispatch.py
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 1,2,12 --out results/dispatch-verify.json -- --dispatch


## Open measurements


In [ ]:
# What the fp32 safety escape costs on case 6 (the seed-robustness trade)
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 6 --out results/case6-fp32-escape.json -- --fp16-max-elements 100000000


In [ ]:
# CUDA graphs on the two untried mid shapes
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 9,10 --out results/pass2-cg-mid.json -- --cuda-graphs


## Summary and bundle


In [ ]:
import json, pathlib
for path in sorted(pathlib.Path('/kaggle/working/transformer-kernel/results').glob('*.json')):
    data = json.loads(path.read_text())
    if 'cases' in data:
        print(f"\n=== {path.name} | {data.get('passthrough_args')}")
        for case in data['cases']:
            acc = case.get('accuracy') or {}
            print(f"  case {case['case']['id']:>2} {case['status']:<16} "
                  f"speedup={case.get('speedup')} max_abs={acc.get('max_abs_error')}")


In [ ]:
!cd /kaggle/working/transformer-kernel && tar czf /kaggle/working/results.tar.gz results/
print('Download results.tar.gz from the notebook Output tab')
